# Hindlimb Model Edits 
+ Use this notebook to keep track of code used to edit the hindlimb model
+ Place functions in the `src` directory and import them in the blocks that need them 
+ Document your changes with markdown cells before the code block in the style of:

## Step 0: Load the model
This step loads the model and sets up the environment for editing.

In [ ]:
import opensim as osim
from src.muscle_utils import attachments_to_csv

# Load the model
model = osim.Model('rat_hindlimb.osim')
attachments_to_csv(model, 'muscle_attachments.csv')

## Step 1: Convert Thelen muscles to Millard
There is an error in the equilibrate muscles function for Thelen muscles, and Millard muscles are more prevalent.

In [ ]:
from src.muscle_utils import model_thelen_to_millard

millard_model = model_thelen_to_millard(model)
# Save the modified model
millard_model.printToXML('rat_hindlimb_millard.osim')

## Step 2: Analyze muscle lengths and moment arms
Create plots of the muscle lengths and moment arms to find discontinuities or other potential errors.

In [ ]:
from src.musculoskeletal_graph import MusculoskeletalGraph

millard_tsl_model = millard_model.clone()
graph = MusculoskeletalGraph(millard_tsl_model)

# Iterate through the range of motion to compute lengths and moment arms for all muscles
results = graph.muscle_rom_analysis(min_points=1000)

In [ ]:
import plotly.graph_objects as go
import os
# import plotly.io as pio
# pio.renderers.default = "notebook"
figure_dir = 'figs'

# Plot lengths and moment arms to identify problems
for muscle_name, muscle_data in results.items():
    # Plot parallel coordinates of muscle lengths
    fig1 = go.Figure(data=
        go.Parcoords(
            line=dict(color=muscle_data['length'], colorscale='Viridis', showscale=True),
            dimensions=[dict(
                label=col,
                values=muscle_data[col],
            )
            for col in muscle_data.columns if not 'moment_arm' in col]
        )   
    )   
    fig1.update_layout(title=muscle_name + ' Lengths')
    # fig1.write_html(f"{figure_dir}/{muscle_name}_length.html")
    if not os.path.exists(f"{figure_dir}/{muscle_name}"):
        os.makedirs(f"{figure_dir}/{muscle_name}")
    fig1.write_image(f"{figure_dir}/{muscle_name}/{muscle_name}_length.png")

    moment_arm_cols = [col for col in muscle_data.columns if 'moment_arm' in col]
    coord_cols = [col for col in muscle_data.columns if not 'moment_arm' in col and col != 'length']
    for moment_arm in moment_arm_cols:
        # Plot scatter plot of moment arm versus singular coordinate
        for coord in coord_cols:
            fig2 = go.Figure(data=
                go.Scatter(
                    x=muscle_data[coord],
                    y=muscle_data[moment_arm],
                    mode='markers',
                    marker=dict(color=muscle_data[moment_arm], colorscale='Viridis', showscale=True),
                )
            )
            fig2.update_layout(title=f'{muscle_name} {moment_arm} vs {coord}')
            # fig2.write_html(f"{figure_dir}/{muscle_name}_{moment_arm}_vs_{coord}.html")
            fig2.write_image(f"{figure_dir}/{muscle_name}/{muscle_name}_{moment_arm}_vs_{coord}.png")

## Step 3: Rough estimate tendon slack lengths
This is done initially so that the model can be opened in OpenSim Creator for easier muscle editing. It will need to be redone later to account for changes to moment arms. This method is currently using the whole range of motion of each joint, but you could also specify coordinate combinations that you expect to see in the analyzed motion. Using the below method, you can catch most issues, but if you want to just focus on a specific motion, it might be unnecessary.

In [ ]:
from src.muscle_utils import optimize_model_tsl

results = graph.muscle_rom_analysis(min_points=10)
millard_tsl_model: osim.Model = optimize_model_tsl(millard_tsl_model, results)

millard_tsl_model.printToXML('rat_hindlimb_millard_tsl.osim')

## Potentially problematic muscles
### BFa 
![BFa Length](./figs/BFa/BFa_length.png)

### CF
![CF Length](./figs/CF/CF_length.png)

### STp
![STp Length](./figs/STp/STp_length.png)

### TA
![TA Length](./figs/TA/TA_length.png)

### VL
![VL Length](./figs/VL/VL_length.png)

Because of the hard-coded value in [OpenSim's WrapCylinder.cpp code](https://github.com/opensim-org/opensim-core/blob/f9cd558ec3ea99dda206e5e412e62e23cf19bd7e/OpenSim/Simulation/Wrap/WrapCylinder.cpp#L668):

```c++
// Each muscle segment on the surface of the cylinder should be
// 0.002 meters long. This assumes the model is in meters, of course.
int numWrapSegments = (int)(aWrapResult.wrap_path_length / 0.002);
if (numWrapSegments < 1) numWrapSegments = 1;
```
we need to modify the wrapping surfaces to prevent discontinuities in muscle lengths that appear in the range of motion of the rats during normal gait

We plan to use the attachment points presented in Young's work which build upon Johnson and Greene's original anatomical text. However, the coordinate systems are defined differently and the scale is different. We will need to adjust the values accordingly. To achieve this, we will calculate the necessary transformation matrices and apply them to the data points.

This can be done by determining the homogenous transformation matrices for each body coordinate system and transforming the values from Young's coordinate system to Johnson model's coordinate system. The transformation matrices will be derived based on the anatomical landmarks defined in both coordinate systems, ensuring accurate mapping of the data points. 

One method to achieve this is to use mesh registration.

We utilize the `open3d` library to facilitate the registration process, allowing us to align the meshes accurately based on the defined anatomical landmarks. Additionally, we implement a function to compute the transformation matrices based on the identified attachment points, ensuring that the registration process is robust and reliable.

In [ ]:
from src.registration import register_meshes, convert_points_between_meshes

# Define the source, target, and output files
stuff = {
    'pelvis': {
        'source': 'pelvis_r_young.stl',
        'target': 'pelvis.stl',
        'output': 'pelvis_y2j.stl'
    },
    'femur': {
        'source': 'femur.stl',
        'target': 'femur.stl',
        'output': 'femur_registration.stl'
    },
}
# Store the transforms


In [ ]:
# Update the model with the new meshes
import opensim as osim
attached_geom = body.get_attached_geometry(0)

In [ ]:
import numpy as np




# TODO: read this from csv file
muscle_points = {
    'BFa': {
        'points':[{
            'type': 'origin',
            'frame': 'pelvis',
            'coordinates': np.array([19.501, -0.571, -5.385], dtype=float),
        },
        {
            'type': 'via',
            'frame': 'pelvis',
            'coordinates': np.array([19.878, -6.071, -2.285], dtype=float),
        },      
        {
            'type': 'via',
            'frame': 'pelvis',
            'coordinates': np.array([19.844, -8.547, 1.219], dtype=float),
        },             
        {
            'type': 'insertion',
            'frame': 'tibia',
            'coordinates': np.array([1.844, -8.961, 2.876], dtype=float),
        }]        
    },
    'CF': {
        'points':[{
            'type': 'origin',
            'frame': 'pelvis',
            'coordinates': np.array([14.103, -1.194, -6.141], dtype=float),
        },
        {
            'type': 'via',
            'frame': 'pelvis',
            'coordinates': np.array([13.65, -5.582, -2.7], dtype=float),
        },      
        {
            'type': 'via',
            'frame': 'pelvis',
            'coordinates': np.array([12.48, -8.318, 0.92], dtype=float),
        },             
        {
            'type': 'insertion',
            'frame': 'femur',
            'coordinates': np.array([-8.599, -4.649, 28.716], dtype=float),
        }]        
    },
    'STp': {
        'points':[{
            'type': 'origin',
            'frame': 'pelvis',
            'coordinates': np.array([22.152, -5.93, 0.642], dtype=float),
        },            
        {
            'type': 'insertion',
            'frame': 'tibia',
            'coordinates': np.array([-2.449, -13.114, -3.591], dtype=float),
        }]        
    }  
}



# Transform the muscle attachment points from the source mesh to the target mesh
# Save the transformed attachment points to a CSV file

In [ ]:
# Update the model with the new muscle paths

## Final Step: Create Bilateral Model

In [ ]:
# from src import mirror_hindlimb
# #TODO: Implement the mirror_hindlimb function to create a bilateral model
# bilateral_model : osim.Model = mirror_hindlimb(model)

# # Save the mirrored model
# bilateral_model.printToXML('rat_hindlimb_bilateral.osim')